# Example 12 — Couette & Poiseuille flow: the "hello world" of viscous fluids

Steady, fully-developed flow between parallel plates reduces Navier–Stokes to one ODE:
$$\mu\,\frac{d^2u}{dy^2} = \frac{dp}{dx} = G\ (\text{const}),\qquad u(0)=0,\quad u(H)=U_{wall}$$
with exact solution ($\mu = H = 1$):
$$u(y) = U_{wall}\,y + \tfrac{G}{2}\,y(y-1).$$

Three classic cases from every fluids course:
- **Couette** ($U=1, G=0$): pure shear, linear profile;
- **Poiseuille** ($U=0, G=-8$): pressure-driven, parabola with $u_{max}=1$ at the centre;
- **Combined** ($U=1, G=-8$): superposition — the linearity of this reduced problem, visible.

**PINN design (deliberately minimal):** hard BCs via the trial function
$$u(y) = U\,y + y(1-y)\,N(y)$$
so $u(0)=0$ and $u(1)=U$ hold *exactly* and the loss is the pure residual
$\|u'' - G\|^2$ — no weights, no tuning, nothing to balance. Converges to ~1e-4 in
seconds. This is the cleanest possible demonstration of the hard-constraint trick from
Example 11.

> Runs in seconds on CPU or GPU.

In [ ]:
# Cell 1 -- One tiny PINN per case; hard BCs; pure residual loss
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

CASES = {  # name: (U_wall, G = dp/dx)
    'Couette':    (1.0,  0.0),
    'Poiseuille': (0.0, -8.0),   # G = -8  ->  u_max = 1 at the centre
    'Combined':   (1.0, -8.0),
}
def u_exact(y, U, G): return U*y + (G/2)*y*(y-1)

yg = torch.linspace(0, 1, 401, device=device).reshape(-1, 1)
plt.figure(figsize=(7.5, 5))
for name, (U, G) in CASES.items():
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(),
                        nn.Linear(32, 32), nn.Tanh(),
                        nn.Linear(32, 1)).to(device)
    u_trial = lambda y: U*y + y*(1-y)*net(y)      # u(0)=0, u(1)=U — EXACT
    opt = torch.optim.Adam(net.parameters(), 2e-3)
    t0 = time.perf_counter()
    for e in range(2000):
        opt.zero_grad()
        y = torch.rand(512, 1, device=device).requires_grad_(True)
        u = u_trial(y)
        uy  = torch.autograd.grad(u,  y, torch.ones_like(u),  create_graph=True)[0]
        uyy = torch.autograd.grad(uy, y, torch.ones_like(uy), create_graph=True)[0]
        ((uyy - G)**2).mean().backward(); opt.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    with torch.no_grad():
        up = u_trial(yg).cpu().numpy().ravel()
    ue = u_exact(yg.cpu().numpy().ravel(), U, G)
    err = np.sqrt(np.mean((up-ue)**2))
    print(f'{name:11s}: L2 = {err:.2e}   ({time.perf_counter()-t0:.1f} s)')
    p = plt.plot(ue, yg.cpu().numpy().ravel(), lw=2.4, alpha=.55)
    plt.plot(up, yg.cpu().numpy().ravel(), '--', color=p[0].get_color(), lw=1.5,
             label=f'{name} (L2={err:.1e})')
plt.xlabel('u(y)'); plt.ylabel('y'); plt.grid(alpha=.3); plt.legend()
plt.title('Channel flows: solid = exact, dashed = PINN')
plt.tight_layout(); plt.show()

## Observations (for fluid-dynamics notes)

- **The pressure gradient is a source term.** In the residual `uyy - G`, the constant
  $G = dp/dx$ plays the same role a body force would — the natural place to start before
  pressure becomes an unknown *field* (Examples 13–14).
- **Hard BCs leave nothing to tune.** With $u = Uy + y(1-y)N(y)$ the loss has exactly one
  term. Compare: every soft-BC notebook in this kit needed a weight (10, 20, 100…) chosen
  by hand. When you *can* build the BCs in, do.
- **Superposition, visible.** The combined profile is the sum of the other two — because
  the reduced equation is linear, even though full NS is not. A nice check: sum the two
  trained PINN profiles and compare to the third.
- **This is the right first NS example for notes:** every term of the momentum equation is
  identifiable ($\mu u''$ diffusion, $G$ forcing; convection identically zero by
  fully-developed symmetry).

**Experiments to try:** make $G$ a network *input* (Example 5's surrogate trick) — one net
for every pressure gradient; solve the inverse problem: given 5 noisy velocity samples,
recover $G$ (Example 2's trick — 3 lines to change); replace the hard-BC trial with soft
BC weights and watch the accuracy drop for the same budget.